In [ ]:
import sys, json, pathlib
sys.path.insert(0, str(pathlib.Path("..").resolve() / "src"))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from qaoa_rw import data, qubo, qaoa, baselines, metrics

RESULTS, FIGS = pathlib.Path("../results"), pathlib.Path("../figures")
RESULTS.mkdir(exist_ok=True); FIGS.mkdir(exist_ok=True)
SEED = 0
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


In [ ]:
N_QUBITS, K = 12, 5
ALPHA, PENALTY = 0.5, 2.0

X, y, names, rel, red = data.load_problem(N_QUBITS, seed=SEED)
pd.DataFrame({"variable": names, "relevancia": rel.round(3)})


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].barh(names[::-1], rel[::-1], color="#2a6f97")
ax[0].set_title("Relevancia (información mutua normalizada)")
im = ax[1].imshow(red, cmap="viridis", vmin=0, vmax=1)
ax[1].set_xticks(range(N_QUBITS)); ax[1].set_yticks(range(N_QUBITS))
ax[1].set_title("Redundancia |corr|"); plt.colorbar(im, ax=ax[1])
plt.tight_layout(); plt.savefig(FIGS / "01_problem.png", dpi=150); plt.show()


In [ ]:
Q, offset = qubo.feature_selection_qubo(rel, red, K, ALPHA, PENALTY)
bits = qubo.all_bitstrings(N_QUBITS)
E = qubo.energies(Q, offset, bits)
opt = int(np.argmin(E))
feasible = bits.sum(axis=1) == K

print(f"Bitstrings: {len(E)} | factibles (|x|={K}): {feasible.sum()} ({feasible.mean():.1%})")
print("Óptimo exacto:", [n for n, b in zip(names, qubo.index_to_mask(opt, N_QUBITS)) if b], f"| E = {E[opt]:.4f}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(E[~feasible], bins=60, alpha=.6, label="infactibles", color="#adb5bd")
ax.hist(E[feasible], bins=60, alpha=.9, label=f"factibles (|x|={K})", color="#2a6f97")
ax.axvline(E[opt], color="crimson", ls="--", label="óptimo")
ax.set(xlabel="energía", ylabel="# bitstrings", title="Paisaje de energías"); ax.legend()
plt.tight_layout(); plt.savefig(FIGS / "01_energy_landscape.png", dpi=150); plt.show()


In [ ]:
h, J, const = qubo.qubo_to_ising(Q, offset)
assert np.allclose(E, qubo.ising_energies(h, J, const, N_QUBITS)), "Ising y QUBO deben coincidir"
print(f"Ising OK · términos Z: {np.count_nonzero(h)} · términos ZZ: {len(J)}")


In [ ]:
np.savez(RESULTS / f"problem_n{N_QUBITS}_k{K}.npz", Q=Q, offset=offset, rel=rel, red=red, names=np.array(names))
print("Guardado:", RESULTS / f"problem_n{N_QUBITS}_k{K}.npz")
